## Open notebook in:
| Colab                                 
:-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nicolepcx/transformers-the-definitive-guide/blob/master/CH06/ch06_Kimi_audio_transcribe_meeting.ipynb)                                             

# About this Notebook

This notebook demonstrates a complete pipeline for **multi-speaker diarization and transcription** using state-of-the-art audio foundation models. The goal is to identify who is speaking when, extract individual speaker segments, and generate corresponding transcriptions using a multimodal large language model.

### Steps Included:

1. **Dataset Loading**:
   The notebook uses the `concatenated_librispeech` dataset from Hugging Face, which combines audio samples containing multiple speakers into longer clips. You can also replace this with your own local audio files.

2. **Speaker Diarization**:
   A pre-trained diarization pipeline from `pyannote.audio` is used to segment the audio by speaker. Each segment is annotated with speaker identity (e.g., `SPEAKER_00`) along with start and end timestamps.

3. **Segment Extraction**:
   Once diarized, the notebook slices the raw waveform into speaker-specific segments. Each audio chunk is saved as an individual `.wav` file for downstream processing.

4. **Model Setup (Kimi-Audio)**:
   The [Kimi-Audio-7B-Instruct](https://huggingface.co/moonshotai/Kimi-Audio-7B-Instruct) model is loaded using the `kimia_infer` library. This model is designed for **instruction-following audio-text tasks**, including transcription, summarization, and Q\&A over audio.

5. **Multimodal Inference**:
   Each speaker segment is paired with a prompt, asking the model to transcribe the speech. The model is run iteratively over all segments to produce labeled transcriptions.

6. **Structured Output**:
   Final results are printed with speaker identity, timestamp boundaries, and transcribed text—providing a detailed, readable transcription of a multi-speaker conversation.

This notebook highlights the power of **hybrid pipelines** that combine **traditional signal processing (for diarization)** with **LLMs capable of multimodal understanding**. It enables detailed, speaker-specific insights from raw audio, suitable for tasks like meeting transcription, podcast summarization, or conversation analysis.



# Installs

In [ ]:
!pip install pyannote.audio -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 893.7/893.7 kB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.1/52.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.8/127.8 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4

In [ ]:
!pip install --upgrade datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
db-dtypes 1.6.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.3 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1

In [ ]:
!pip install git+https://github.com/MoonshotAI/Kimi-Audio.git

  Cloning https://github.com/MoonshotAI/Kimi-Audio.git to /tmp/pip-req-build-v4ajzb1d
  Running command git clone --filter=blob:none --quiet https://github.com/MoonshotAI/Kimi-Audio.git /tmp/pip-req-build-v4ajzb1d
  Resolved https://github.com/MoonshotAI/Kimi-Audio.git to commit 349251e1d8f4f98d58fda59246381faecd7392e0
  Running command git submodule update --init --recursive -q
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 109.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.1 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 7.2 MB/s eta 0:00:00
Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (180 kB)
   ━━━━━━━━

In [ ]:
!pip uninstall -y flash-attn
!pip install flash-attn==2.7.2post1


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 42.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


# Imports

In [ ]:
from datasets import load_dataset
from pyannote.audio import Pipeline
import torch
import soundfile as sf
import os
from kimia_infer.api.kimia import KimiAudio


# Load audio from LibriSpeech dataset or local file

In [ ]:
dataset = load_dataset("sanchit-gandhi/concatenated_librispeech")
#sample = dataset[0]



In [ ]:
dataset

### Att: you need to request access to the models for the pyannote

In [ ]:
# Diarization pipeline
diarization_pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization",
    revision="2.1",
    token="HUGGINGFACE_ACCESS_TOKEN",
)
dataset = load_dataset("sanchit-gandhi/concatenated_librispeech", split="train")


# Slice audio

In [ ]:
sample = dataset[0]
audio_array = sample["audio"]["array"]
sr = sample["audio"]["sampling_rate"]

waveform = torch.tensor(audio_array[None, :]).float()
annotation = diarization_pipeline({"waveform": waveform, "sample_rate": sr})

segments = []
for turn, _, speaker in annotation.itertracks(yield_label=True):
    segments.append({
        "start": turn.start,
        "end": turn.end,
        "speaker": speaker
    })

# Now you can slice:
speaker_segments = []
for seg in segments:
    start = int(seg["start"] * sr)
    end = int(seg["end"] * sr)
    audio_chunk = audio_array[start:end]
    speaker_segments.append({
        "speaker": seg["speaker"],
        "start": seg["start"],
        "end": seg["end"],
        "audio": audio_chunk
    })


## Show segments

In [ ]:
speaker_segments

# Save Speaker Segments as Individual Audio Files

This section creates a temporary folder and saves each diarized speaker segment as a separate `.wav` file for later transcription.


In [ ]:

# Create temp folder
os.makedirs("tmp_segments", exist_ok=True)

for i, seg in enumerate(speaker_segments):
    filename = f"tmp_segments/speaker_{i}.wav"
    sf.write(filename, seg["audio"], sr)
    speaker_segments[i]["file_path"] = filename


# Load Kimi-Audio-7B-Instruct for Multimodal Inference

Here, the KimiAudio model is loaded with detokenizer support and inference parameters are defined for both audio and text modalities.


In [ ]:

model_path = "moonshotai/Kimi-Audio-7B-Instruct"
model = KimiAudio(model_path=model_path, load_detokenizer=True)

sampling_params = {
    "audio_temperature": 0.8,
    "audio_top_k": 10,
    "text_temperature": 0.0,
    "text_top_k": 5,
    "audio_repetition_penalty": 1.0,
    "audio_repetition_window_size": 64,
    "text_repetition_penalty": 1.0,
    "text_repetition_window_size": 16,
}


# Multi-Speaker Transcription Using Joint Audio & Text Prompts

All segmented speaker audio clips are paired with time-based questions and passed to the model as a **batched message sequence** to generate a full conversation-level transcript.


In [ ]:
messages = []
for seg in speaker_segments:
    # Add optional prompt to control output
    messages.append({
        "role": "user",
        "message_type": "text",
        "content": f"What is {seg['speaker']} saying between {seg['start']:.1f}s and {seg['end']:.1f}s?"
    })
    messages.append({
        "role": "user",
        "message_type": "audio",
        "content": seg["file_path"]
    })


# Detailed Per-Speaker Transcription (Segment-Wise Loop)

Each speaker segment is processed independently using a prompt + audio pair to generate **fine-grained transcriptions** with speaker labels and timestamps.


In [ ]:
_, text_output = model.generate(messages, **sampling_params, output_type="text")
print(">>> Transcribed Multi-speaker Output:\n")
print(text_output)


In [ ]:
transcriptions = []

for seg in speaker_segments:
    msgs = [
        {"role": "user", "message_type": "text", "content": f"Transcribe what {seg['speaker']} says:"},
        {"role": "user", "message_type": "audio", "content": seg["file_path"]}
    ]
    _, output = model.generate(msgs, **sampling_params, output_type="text")
    transcriptions.append({
        "speaker": seg["speaker"],
        "start": seg["start"],
        "end": seg["end"],
        "text": output
    })


# Print Final Transcription Results with Speaker and Time Annotations

The final output is printed in a structured format showing **who spoke when** and **what was said**, enabling easy inspection of multi-speaker conversations.


In [ ]:
for t in transcriptions:
    print(f"{t['speaker']} ({t['start']:.1f}s - {t['end']:.1f}s): {t['text']}\n")
